# FINN: ONNX to Bitstream Conversion for W4A4 Road Sign Classifier

This notebook provides a comprehensive pipeline to convert your W4A4 quantized road sign classifier ONNX model to a FINN bitstream for deployment on PYNQ-Z2.

## Prerequisites
- FINN Docker container running
- Exported QONNX model: `road_sign_classifier_finn_w4a4.onnx`
- Vivado/Vitis license configured
- PYNQ-Z2 board files installed

## Overview
1. Setup build environment
2. Create build configuration
3. Create folding configuration
4. Run FINN compiler
5. Deploy to PYNQ-Z2

## Step 1: Environment Setup and Imports

In [1]:
import os
import json
import shutil
from pathlib import Path
import numpy as np
from datetime import datetime

# FINN imports
from finn.builder.build_dataflow import build_dataflow_cfg
from finn.builder.build_dataflow_config import (
    DataflowBuildConfig,
    ShellFlowType,
    DataflowOutputType,
    VerificationStepType,
    AutoFIFOSizingMethod
)
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.util.basic import qonnx_make_model

print("🚀 FINN ONNX to Bitstream Converter")
print("=" * 60)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Working directory: {os.getcwd()}")

🚀 FINN ONNX to Bitstream Converter
Timestamp: 2025-05-27 06:26:43
Working directory: /home/rdl-ws2/Desktop/Project/finn/notebooks/Project


## Step 2: Configure Build Paths

In [2]:
# Define paths - MODIFY THESE AS NEEDED
BUILD_DIR = "/tmp/finn_road_sign_build"  # Main build directory
MODEL_PATH = "models/road_sign_minimal_finn_w4a4.onnx"  # Your QONNX model
OUTPUT_DIR = "finn_outputs"  # Where to save final outputs

# Board configuration
TARGET_BOARD = "Pynq-Z2"  # Options: Pynq-Z1, Pynq-Z2, Ultra96, ZCU104, etc.
TARGET_CLOCK_NS = 10.0  # Target clock period in nanoseconds (10ns = 100MHz)
TARGET_FPS = 1000  # Target frames per second for auto-folding

# Create directories
os.makedirs(BUILD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy model to build directory
model_in_build = os.path.join(BUILD_DIR, "model.onnx")
if os.path.exists(MODEL_PATH):
    shutil.copy(MODEL_PATH, model_in_build)
    print(f"✅ Model copied to: {model_in_build}")
else:
    print(f"❌ Model not found at: {MODEL_PATH}")
    print("Please ensure your QONNX model is exported first!")

✅ Model copied to: /tmp/finn_road_sign_build/model.onnx


## Step 3: Create Folding Configuration

The folding configuration controls the parallelization of your design. Higher PE/SIMD values mean more parallel processing but more FPGA resources.

In [3]:
# Manual folding configuration for W4A4 road sign classifier
# This provides a balanced design for PYNQ-Z2
folding_config = {
#   "Defaults": {
#     "SIMD": [2, "all"],
#     "PE": [8, "all"],
#     "ram_style": ["block", "all"],
#     "resType": ["lut", "all"]
#   },
#   "StreamingFCLayer_0": {
#     "PE": 4,
#     "SIMD": 3,
#     "ram_style": "distributed",
#     "resType": "lut"
#   },
#   "StreamingFCLayer_1": {
#     "PE": 8,
#     "SIMD": 8,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "StreamingFCLayer_2": {
#     "PE": 8,
#     "SIMD": 8,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "StreamingFCLayer_3": {
#     "PE": 16,
#     "SIMD": 16,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "StreamingFCLayer_4": {
#     "PE": 16,
#     "SIMD": 16,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "StreamingFCLayer_5": {
#     "PE": 32,
#     "SIMD": 32,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "StreamingFCLayer_6": {
#     "PE": 43,
#     "SIMD": 32,
#     "ram_style": "distributed",
#     "resType": "lut"
#   },
#   "MVAU_hls_5": {
#     "SIMD": 4,
#     "PE": 16,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "StreamingFIFO_0": {"depth": 128},
#   "StreamingFIFO_1": {"depth": 256},
#   "StreamingFIFO_2": {"depth": 256},
#   "StreamingFIFO_3": {"depth": 512},
#   "StreamingFIFO_4": {"depth": 512},
#   "StreamingFIFO_5": {"depth": 256},
#   "StreamingFIFO_6": {"depth": 128}
# }

#   "Defaults": {
#     "SIMD": [2, ["StreamingFCLayer_Batch", "MVAU"]],
#     "PE": [8, ["StreamingFCLayer_Batch", "MVAU"]]
#   },
#   "MVAU_hls_5": {
#     "SIMD": 4,
#     "PE": 16
#   },
#   "StreamingFIFO_0": {"depth": 128},
#   "StreamingFIFO_1": {"depth": 256},
#   "StreamingFIFO_2": {"depth": 256},
#   "StreamingFIFO_3": {"depth": 512},
#   "StreamingFIFO_4": {"depth": 512},
#   "StreamingFIFO_5": {"depth": 256},
#   "StreamingFIFO_6": {"depth": 128}
# }

    
#   "Defaults": {
#     "SIMD": [2, ["StreamingFCLayer_Batch", "MVAU"]],
#     "PE": [8, ["StreamingFCLayer_Batch", "MVAU"]]
#   },
#   "MVAU_hls_4": {
#     "SIMD": 2,
#     "PE": 8,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "MVAU_hls_5": {
#     "SIMD": 4,
#     "PE": 16,
#     "ram_style": "block",
#     "resType": "lut"
#   },
#   "StreamingFIFO_0": {"depth": 128},
#   "StreamingFIFO_1": {"depth": 256},
#   "StreamingFIFO_2": {"depth": 256},
#   "StreamingFIFO_3": {"depth": 512},
#   "StreamingFIFO_4": {"depth": 512},
#   "StreamingFIFO_5": {"depth": 256},
#   "StreamingFIFO_6": {"depth": 128}
# }

    
        "Defaults": {
            "SIMD": [2, ["StreamingFCLayer_Batch", "MVAU"]],
            "PE": [4, ["StreamingFCLayer_Batch", "MVAU"]],
            "ram_style": ["block", ["StreamingFCLayer_Batch", "MVAU"]],
            "resType": ["lut", ["StreamingFCLayer_Batch", "MVAU"]]
        },
        # Block 1 layers
        "StreamingFCLayer_0": {
            "PE": 4,
            "SIMD": 3,
            "ram_style": "distributed"
        },
        # Block 2 layers
        "StreamingFCLayer_1": {
            "PE": 4,
            "SIMD": 4,
            "ram_style": "block"
        },
        # Block 3 layers
        "StreamingFCLayer_2": {
            "PE": 8,
            "SIMD": 8,
            "ram_style": "block"
        },
        # Classifier layers
        "StreamingFCLayer_3": {
            "PE": 16,
            "SIMD": 16,
            "ram_style": "block"
        },
        # MVAU nodes
        "MVAU_hls_0": {
            "SIMD": 3,
            "PE": 4
        },
        "MVAU_hls_1": {
            "SIMD": 4,
            "PE": 4
        },
        "MVAU_hls_2": {
            "SIMD": 8,
            "PE": 8
        },
        # FIFO depths - keep small to reduce resource usage
        "StreamingFIFO_0": {"depth": 64},
        "StreamingFIFO_1": {"depth": 64},
        "StreamingFIFO_2": {"depth": 128},
        "StreamingFIFO_3": {"depth": 128},
        "StreamingFIFO_4": {"depth": 256},
        "StreamingFIFO_5": {"depth": 256}
    }


# Save folding configuration
folding_config_path = os.path.join(BUILD_DIR, "folding_config.json")
with open(folding_config_path, 'w') as f:
    json.dump(folding_config, f, indent=2)
print(f"✅ Folding config saved to: {folding_config_path}")

✅ Folding config saved to: /tmp/finn_road_sign_build/folding_config.json


## Step 4: Create Build Configuration

This configures the entire FINN build flow.

In [4]:
# Create FINN build configuration
build_cfg = DataflowBuildConfig(
    # Output directory for this build
    output_dir=os.path.join(BUILD_DIR, f"output_{TARGET_BOARD.lower()}"),
    
    # Target board
    board=TARGET_BOARD,
    shell_flow_type=ShellFlowType.VIVADO_ZYNQ if "Pynq" in TARGET_BOARD else ShellFlowType.VITIS_ALVEO,
    
    # Clock configuration
    synth_clk_period_ns=TARGET_CLOCK_NS,
    
    # Folding configuration
    folding_config_file=folding_config_path,
    
    # Auto-optimization settings
    auto_fifo_depths=True,  # Automatically determine FIFO depths
    auto_fifo_strategy=AutoFIFOSizingMethod.LARGEFIFO_RTLSIM,
    
    # Optional: Use target FPS instead of manual folding
    # target_fps=TARGET_FPS,  # Uncomment to use automatic folding
    
    # Build outputs - what to generate
    generate_outputs=[
        DataflowOutputType.ESTIMATE_REPORTS,      # Resource estimates
        DataflowOutputType.STITCHED_IP,          # Stitched Vivado IP
        DataflowOutputType.RTLSIM_PERFORMANCE,   # Performance metrics
        DataflowOutputType.OOC_SYNTH,            # Out-of-context synthesis
        DataflowOutputType.BITFILE,              # Final bitstream
        DataflowOutputType.PYNQ_DRIVER,          # Python driver
        DataflowOutputType.DEPLOYMENT_PACKAGE,   # Complete deployment package
    ],
    
    # Verification settings (optional but recommended)
    verify_steps=[
        VerificationStepType.QONNX_TO_FINN_PYTHON,
        VerificationStepType.STREAMLINED_PYTHON,
        VerificationStepType.FOLDED_HLS_CPPSIM,
    ],
    
    # RTL simulation settings
    rtlsim_batch_size=1,
    # rtlsim_timeout=1000,
    
    # Save intermediate models for debugging
    save_intermediate_models=True,
)

# Save build configuration
build_cfg_path = os.path.join(BUILD_DIR, "build_config.json")
build_cfg_dict = build_cfg.to_dict()
with open(build_cfg_path, 'w') as f:
    json.dump(build_cfg_dict, f, indent=2)
print(f"✅ Build config saved to: {build_cfg_path}")
print(f"\n📋 Build Configuration:")
print(f"  - Board: {TARGET_BOARD}")
print(f"  - Clock: {1000/TARGET_CLOCK_NS:.1f} MHz")
print(f"  - Auto FIFO sizing: {build_cfg.auto_fifo_depths}")
print(f"  - Outputs: {len(build_cfg.generate_outputs)} products")

✅ Build config saved to: /tmp/finn_road_sign_build/build_config.json

📋 Build Configuration:
  - Board: Pynq-Z2
  - Clock: 100.0 MHz
  - Auto FIFO sizing: True
  - Outputs: 7 products


## Step 5: Create Test Data for Verification (Optional)

Create test input/output for verification during the build process.

In [5]:
# Create test data for verification
test_input = np.random.rand(1, 3, 32, 32).astype(np.float32)
test_input_path = os.path.join(BUILD_DIR, "test_input.npy")
np.save(test_input_path, test_input)

# For expected output, you would need to run inference with your trained model
# For now, we'll create a dummy output
test_output = np.zeros((1, 43), dtype=np.float32)
test_output[0, 0] = 1.0  # Dummy prediction
test_output_path = os.path.join(BUILD_DIR, "test_output.npy")
np.save(test_output_path, test_output)

# Update build config with verification data
build_cfg.verify_input_npy = test_input_path
build_cfg.verify_expected_output_npy = test_output_path

print(f"✅ Test data created:")
print(f"  - Input shape: {test_input.shape}")
print(f"  - Output shape: {test_output.shape}")

✅ Test data created:
  - Input shape: (1, 3, 32, 32)
  - Output shape: (1, 43)


## Step 6: Run FINN Compiler

This is the main compilation step. It will take 30 minutes to several hours depending on your design complexity and selected outputs.

In [7]:
print("\n🔧 Starting FINN compilation...")
print("This will take a while (30 min - 2+ hours). Progress will be shown below.")
print("=" * 60)

try:
    # Load the ONNX model
    model = ModelWrapper(model_in_build)
    print(f"✅ Model loaded: {model_in_build}")
    print(f"  - Input shape: {model.get_tensor_shape(model.graph.input[0].name)}")
    print(f"  - Output shape: {model.get_tensor_shape(model.graph.output[0].name)}")
    
    # Run the build flow
    print("\n🚀 Launching FINN build flow...")
    build_dataflow_cfg(model_in_build, build_cfg)
    
    print("\n✅ FINN compilation completed successfully!")
    
except Exception as e:
    print(f"\n❌ Build failed with error: {e}")
    print("\nCommon issues:")
    print("1. Vivado license not configured")
    print("2. Board files not installed")
    print("3. Insufficient memory/disk space")
    print("4. Model compatibility issues")
    raise


🔧 Starting FINN compilation...
This will take a while (30 min - 2+ hours). Progress will be shown below.
✅ Model loaded: /tmp/finn_road_sign_build/model.onnx
  - Input shape: [1, 3, 32, 32]
  - Output shape: [0, 0]

🚀 Launching FINN build flow...
Building dataflow accelerator from /tmp/finn_road_sign_build/model.onnx
Intermediate outputs will be generated in /home/rdl-ws2/tmp_finn
Final outputs will be generated in /tmp/finn_road_sign_build/output_pynq-z2
Build log is at /tmp/finn_road_sign_build/output_pynq-z2/build_dataflow.log
Running step: step_qonnx_to_finn [1/19]
Running step: step_tidy_up [2/19]
Running step: step_streamline [3/19]
Running step: step_convert_to_hw [4/19]
Running step: step_create_dataflow_partition [5/19]
Running step: step_specialize_layers [6/19]
Running step: step_target_fps_parallelization [7/19]
Running step: step_apply_folding_config [8/19]
Running step: step_minimize_bit_width [9/19]
Running step: step_generate_estimate_reports [10/19]
Running step: step

## Step 7: Examine Build Results

In [8]:
# Path to build outputs
output_path = build_cfg.output_dir

print("\n📊 Build Results:")
print("=" * 60)

# Check for generated files
expected_files = {
    "bitfile": "bitfile/finn-accel.bit",
    "hardware handoff": "bitfile/finn-accel.hwh",
    "driver": "driver/driver.py",
    "stitched IP": "stitched_ip/finn_design_wrapper.v",
    "deployment package": "deploy/",
    "reports": "report/"
}

for desc, rel_path in expected_files.items():
    full_path = os.path.join(output_path, rel_path)
    if os.path.exists(full_path):
        print(f"✅ {desc}: {full_path}")
    else:
        print(f"❌ {desc}: Not found")

# Read performance metrics if available
perf_json = os.path.join(output_path, "report/rtlsim_performance.json")
if os.path.exists(perf_json):
    with open(perf_json, 'r') as f:
        perf = json.load(f)
    print("\n📈 Performance Metrics:")
    print(f"  - Throughput: {perf.get('throughput[images/s]', 'N/A'):.2f} fps")
    print(f"  - Latency: {perf.get('DRAM_in_out_latency[cycles]', 'N/A')} cycles")
    print(f"  - Clock: {perf.get('fclk[mhz]', 'N/A')} MHz")

# Read resource estimates if available
synth_json = os.path.join(output_path, "report/ooc_synth_and_timing.json")
if os.path.exists(synth_json):
    with open(synth_json, 'r') as f:
        synth = json.load(f)
    print("\n💾 Resource Utilization:")
    print(f"  - LUTs: {synth.get('LUT', 'N/A')}")
    print(f"  - FFs: {synth.get('FF', 'N/A')}")
    print(f"  - BRAM: {synth.get('BRAM_18K', 'N/A')}")
    print(f"  - DSP: {synth.get('DSP', 'N/A')}")
    print(f"  - Max frequency: {synth.get('fmax_mhz', 'N/A'):.1f} MHz")


📊 Build Results:
✅ bitfile: /tmp/finn_road_sign_build/output_pynq-z2/bitfile/finn-accel.bit
✅ hardware handoff: /tmp/finn_road_sign_build/output_pynq-z2/bitfile/finn-accel.hwh
✅ driver: /tmp/finn_road_sign_build/output_pynq-z2/driver/driver.py
❌ stitched IP: Not found
✅ deployment package: /tmp/finn_road_sign_build/output_pynq-z2/deploy/
✅ reports: /tmp/finn_road_sign_build/output_pynq-z2/report/

📈 Performance Metrics:
  - Throughput: 645.64 fps
  - Latency: N/A cycles
  - Clock: 100.0 MHz

💾 Resource Utilization:
  - LUTs: 9729.0
  - FFs: 13843.0
  - BRAM: 4.0
  - DSP: 0.0
  - Max frequency: 108.7 MHz


## Step 8: Prepare Deployment Package

In [9]:
# Copy deployment files to output directory
deployment_dir = os.path.join(OUTPUT_DIR, f"road_sign_classifier_{TARGET_BOARD.lower()}")
os.makedirs(deployment_dir, exist_ok=True)

# Files to copy for deployment
deployment_files = [
    ("bitfile/finn-accel.bit", "finn-accel.bit"),
    ("bitfile/finn-accel.hwh", "finn-accel.hwh"),
    ("driver/driver.py", "driver.py"),
    ("driver/validate.py", "validate.py"),
]

print("\n📦 Creating deployment package...")
for src, dst in deployment_files:
    src_path = os.path.join(output_path, src)
    dst_path = os.path.join(deployment_dir, dst)
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
        print(f"✅ Copied: {dst}")
    else:
        print(f"⚠️  Not found: {src}")

# Create a simple README file
readme_content = f"""
# Road Sign Classifier FINN Deployment Package

This package contains the FINN-generated accelerator for the road sign classifier.

## Contents
- finn-accel.bit: FPGA bitstream
- finn-accel.hwh: Hardware handoff file
- driver.py: Python driver for inference
- validate.py: Validation script

## Deployment Instructions
1. Copy this entire folder to your PYNQ-Z2 board
2. Connect to the board via Jupyter notebook
3. Run the example in driver.py

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Target: {TARGET_BOARD}
Clock: {1000/TARGET_CLOCK_NS:.1f} MHz
"""

with open(os.path.join(deployment_dir, "README.md"), 'w') as f:
    f.write(readme_content)
print(f"✅ Created: README.md")

# Create a simple example script
example_content = """
# Road Sign Classifier Example
import numpy as np
from PIL import Image
import time
from driver import RoadSignClassifier

# Initialize the classifier
classifier = RoadSignClassifier()

# Load and preprocess an image (replace with your image path)
def preprocess_image(image_path):
    img = Image.open(image_path).resize((32, 32))
    img_np = np.array(img).astype(np.float32) / 255.0
    # Convert to NCHW format (batch, channels, height, width)
    img_np = np.transpose(img_np, (2, 0, 1))[np.newaxis, :]
    return img_np

# Run inference
try:
    # Replace with your image path
    image_path = "your_road_sign_image.jpg"  
    
    # Try with a sample random input if no image is available
    try:
        input_data = preprocess_image(image_path)
        print(f"Loaded image from {image_path}")
    except:
        print("No image found, using random test data")
        input_data = np.random.rand(1, 3, 32, 32).astype(np.float32)
    
    # Run inference and measure time
    print("Running inference...")
    start_time = time.time()
    result = classifier.predict(input_data)
    inference_time = (time.time() - start_time) * 1000  # ms
    
    # Get the predicted class
    predicted_class = np.argmax(result)
    
    print(f"Predicted class: {predicted_class}")
    print(f"Inference time: {inference_time:.2f} ms")
    
except Exception as e:
    print(f"Error during inference: {e}")
finally:
    # Clean up
    classifier.cleanup()
"""

with open(os.path.join(deployment_dir, "example.py"), 'w') as f:
    f.write(example_content)
print(f"✅ Created: example.py")

# Create a zip file of the deployment package
import zipfile
zip_path = os.path.join(OUTPUT_DIR, f"road_sign_classifier_{TARGET_BOARD.lower()}.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(deployment_dir):
        for file in files:
            zipf.write(os.path.join(root, file), 
                      os.path.relpath(os.path.join(root, file), 
                                     os.path.join(deployment_dir, '..')))

print(f"\n✅ Deployment package created: {zip_path}")
print(f"\n🎉 FINN bitstream generation complete! Your accelerator is ready for deployment.")


📦 Creating deployment package...
✅ Copied: finn-accel.bit
✅ Copied: finn-accel.hwh
✅ Copied: driver.py
✅ Copied: validate.py
✅ Created: README.md
✅ Created: example.py

✅ Deployment package created: finn_outputs/road_sign_classifier_pynq-z2.zip

🎉 FINN bitstream generation complete! Your accelerator is ready for deployment.
